# ChestX-ray14 classification: techniques from the papers

This notebook builds a **14-label** chest-radiograph classifier using NIH ChestX-ray14. One image can have several findings at once, so the model predicts each finding independently rather than choosing one mutually exclusive class.

It is an educational implementation based on several papers, not a reproduction of any paper's reported result. The NIH labels were mined from radiology reports and can be noisy; predictions and attention maps are not a diagnosis or validated localization.

## What I read, and what this notebook uses

| Paper | Technique or lesson | How it informs this notebook |
|---|---|---|
| [Wang et al., CVPR 2017 — ChestX-ray8](https://openaccess.thecvf.com/content_cvpr_2017/html/Wang_ChestX-ray8_Hospital-Scale_Chest_CVPR_2017_paper.html) | Introduced the large NIH image collection and weakly supervised labels/localization. Labels come from reports, not pixel-level expert annotations. | Treat labels as image-level targets; preserve patient separation; avoid claiming attention maps are proven lesion locations. |
| [Yao et al., 2017 — Learning to diagnose from scratch by exploiting dependencies among labels](https://arxiv.org/abs/1710.10501) | Uses an LSTM to model relationships among the 14 findings. | Motivates the multi-label setup. This notebook does not implement label-sequence modeling; each finding is predicted independently. |
| [Rajpurkar et al., 2017 — CheXNet](https://arxiv.org/abs/1711.05225) | ImageNet-pretrained DenseNet-121; 224×224 images; sigmoid outputs and binary cross-entropy for the 14-label extension; reports per-finding AUROC. | Backbone, image scale, multi-label loss, and per-label evaluation. |
| [Ge et al., 2018 — Chest X-rays Classification: A Multi-Label and Fine-Grained Problem](https://arxiv.org/abs/1807.07247) | Studies a custom multi-label softmax loss and fine-grained/bilinear features to address co-occurring labels and imbalance. | Highlights that ordinary BCE is a baseline, not the only sensible loss. This notebook keeps BCE transparent and offers optional capped positive weights. |
| [Guan & Huang, 2020 — Category-wise Residual Attention Learning](https://doi.org/10.1016/j.patrec.2018.10.027) | Learns category-specific attention so different diseases can use different image evidence. | The head creates one spatial attention map per finding. It is a simplified attention head, not CRAL's residual block. |
| [Wang et al., 2021 — Triple Attention Learning (A³Net)](https://doi.org/10.1016/j.media.2020.101846) | DenseNet-121 with channel-wise, element-wise, and scale-wise attention; uses ChestX-ray14's patient-level official split. | The notebook uses ImageNet-pretrained DenseNet features, a channel gate, category-specific spatial attention, and the official held-out patient split. It does not reproduce A³Net's scale-wise module or ensemble. |

### Notebook workflow

1. Attach the Kaggle dataset, download it with KaggleHub, or select an extracted local copy.
2. Audit labels and image files; use the supplied official train/validation and test image lists.
3. Create a patient-disjoint validation set from the official training patients.
4. Fine-tune DenseNet-121 with category-specific attention and multi-label BCE.
5. Select the checkpoint using validation AUROC, then evaluate once on the untouched test patients.
6. Inspect per-label AUROC / average precision and an attention overlay; save checkpoints and tables to the run folder.

The original archive is large. This notebook expects the images and metadata to be downloaded and extracted beforehand; it does not silently download tens of gigabytes or require Kaggle credentials.

## 1. Install the small dependencies

PyTorch and torchvision are required. Google Colab usually provides them; for a local environment, install a PyTorch build appropriate for your hardware from the [official PyTorch installation page](https://pytorch.org/get-started/locally/).

In [ ]:
%pip install -q pandas scikit-learn matplotlib tqdm pillow kagglehub

In [ ]:
from pathlib import Path
import os, random, json, math, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
)

print('PyTorch:', torch.__version__)
print('Torchvision import succeeded')
print('CUDA available:', torch.cuda.is_available())


## 2. Import ChestX-ray14 from Kaggle

Dataset page: [NIH Chest X-rays on Kaggle](https://www.kaggle.com/datasets/nih-chest-xrays/data). This notebook needs the **full dataset**, including `Data_Entry_2017*.csv`, `train_val_list.txt`, `test_list.txt`, and the PNG images.

Choose one route in the next cell:

- **Kaggle Notebook:** click **Add Input**, attach “NIH Chest X-rays”, and use `DATA_SOURCE="kaggle_input"`. KaggleHub is already authenticated inside a Kaggle Notebook, and attached files are available under `/kaggle/input`.
- **Google Colab or local Jupyter:** keep `DATA_SOURCE="kagglehub"`. KaggleHub downloads the pinned version of the dataset and returns its extracted directory. The full data is large; allow at least 110 GiB free for download, extraction, and cached files.
- **Already downloaded:** set `DATA_SOURCE="local"` and point `LOCAL_DATA_ROOT` at the extracted folder.

If Kaggle requests consent or authentication, sign in to Kaggle and accept the dataset conditions. In Colab, store `KAGGLE_API_TOKEN` in Colab Secrets and read it from the secret store, or use `kagglehub.login()`. Never paste a token into a notebook cell or commit it to a repository. See [KaggleHub authentication and download docs](https://github.com/Kaggle/kagglehub#authenticate) and [Kaggle API docs](https://www.kaggle.com/docs/api).

In [ ]:
# Choose: 'kagglehub', 'kaggle_input', or 'local'.
DATA_SOURCE = 'kagglehub'
KAGGLE_HANDLE = 'nih-chest-xrays/data/versions/3'
LOCAL_DATA_ROOT = Path('/content/nih-chestxray14')  # edit for your extracted local copy

if DATA_SOURCE == 'kagglehub':
    import kagglehub
    DATA_ROOT = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
elif DATA_SOURCE == 'kaggle_input':
    DATA_ROOT = Path('/kaggle/input')
elif DATA_SOURCE == 'local':
    DATA_ROOT = LOCAL_DATA_ROOT
else:
    raise ValueError("DATA_SOURCE must be 'kagglehub', 'kaggle_input', or 'local'")

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset location does not exist: {DATA_ROOT}')
metadata_candidates = sorted(set(DATA_ROOT.rglob('Data_Entry_2017*.csv')))
if len(metadata_candidates) != 1:
    raise FileNotFoundError(
        f'Expected exactly one Data_Entry_2017*.csv under {DATA_ROOT}; '
        f'found {len(metadata_candidates)}. If using Kaggle inputs, set DATA_ROOT to the NIH dataset folder.'
    )
for required_file in ['train_val_list.txt', 'test_list.txt']:
    matches = list(DATA_ROOT.rglob(required_file))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {required_file} under {DATA_ROOT}; found {len(matches)}')
if next(DATA_ROOT.rglob('*.png'), None) is None:
    raise FileNotFoundError(f'No PNG images found under {DATA_ROOT}. Extract all image archives first.')

print('Dataset source:', DATA_SOURCE)
print('Resolved dataset directory:', DATA_ROOT.resolve())
print('Metadata:', metadata_candidates[0])
print('Official split lists found; PNG images found.')


## 3. Configure the new run and save results to Drive

This fresh run is named **chestxray14_densenet_attention_seed42_v2_20261027** and has `RESUME_TRAINING=False`. On Colab, all results go into:

`/content/drive/MyDrive/ChestXrayProject/chestxray14_densenet_attention_seed42_v2_20261027/`

The run folder contains checkpoints, predictions, metrics, a standalone HTML report, `report_tables/`, and `figures/` (300-dpi PNG and PDF). Google Drive must mount successfully to use that location. Outside Colab, the existing local/Kaggle fallback applies. `NUM_WORKERS=0` is kept for your Colab L4 setup.

For a later restart, keep the same name and settings, set `RESUME_TRAINING=True`, and rerun from the top. Checkpoints are saved after each completed epoch. Save the notebook itself separately with **File → Save a copy in Drive**.


In [ ]:
# Set False to save under the current runtime/work directory instead of Google Drive.
SAVE_TO_GOOGLE_DRIVE = True
RUN_NAME = 'chestxray14_densenet_attention_seed42_v2_20261027'
RESUME_TRAINING = False

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0  # keep zero for this Colab L4 setup
EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
VALIDATION_FRACTION = 0.10
USE_POS_WEIGHT = False
MAX_POS_WEIGHT = 10.0

LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural_Thickening', 'Hernia'
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_BASE = Path('/content/drive/MyDrive/ChestXrayProject')
    except ImportError:
        print('Google Colab not detected; saving to the runtime/work directory instead.')
        OUTPUT_BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
else:
    OUTPUT_BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()

OUTPUT_DIR = OUTPUT_BASE / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_CONFIG = {
    'seed': SEED, 'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'validation_fraction': VALIDATION_FRACTION,
    'use_pos_weight': USE_POS_WEIGHT, 'max_pos_weight': MAX_POS_WEIGHT,
    'labels': LABELS, 'dataset_source': DATA_SOURCE, 'dataset_handle': KAGGLE_HANDLE,
    'dataset_root': str(DATA_ROOT.resolve()),
}
run_config_path = OUTPUT_DIR / 'run_config.json'
if run_config_path.exists():
    saved_config = json.loads(run_config_path.read_text(encoding='utf-8'))
    if not RESUME_TRAINING:
        raise FileExistsError(
            f'{OUTPUT_DIR} already has a run. Set RESUME_TRAINING=True to resume with the same settings, '
            'or choose a new RUN_NAME for a fresh run.'
        )
    changed = [key for key, value in RUN_CONFIG.items()
               if key != 'epochs' and saved_config.get(key) != value]
    if changed:
        raise ValueError(f'Cannot resume because these settings changed: {changed}. Use a new RUN_NAME.')
    if EPOCHS < int(saved_config.get('epochs', EPOCHS)):
        raise ValueError('When resuming, EPOCHS must be at least the prior configured epoch count.')
elif RESUME_TRAINING:
    raise FileNotFoundError(f'Cannot resume: no run_config.json found in {OUTPUT_DIR}')

run_config_path.write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')
print('Device:', DEVICE)
print('Run folder:', OUTPUT_DIR.resolve())
print('Resume:', RESUME_TRAINING)

FIGURES_DIR = OUTPUT_DIR / 'figures'
TABLES_DIR = OUTPUT_DIR / 'report_tables'
FIGURES_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(exist_ok=True)


## 4. Read metadata and audit the image collection

The `No Finding` string is not one of the 14 disease outputs: it becomes an all-zero disease vector. That means “no finding was reported in this dataset label,” not proof that the image is clinically normal. Each disease is encoded as 0/1 and multiple positives are allowed.

In [ ]:
csv_candidates = metadata_candidates  # found once in the Kaggle/local import cell above
if len(csv_candidates) != 1:
    raise FileNotFoundError(
        f'Expected exactly one Data_Entry_2017*.csv under {DATA_ROOT}; found {len(csv_candidates)}. '
        'Point DATA_ROOT at one full dataset copy.'
    )
metadata_path = csv_candidates[0]
metadata = pd.read_csv(metadata_path, dtype={'Patient ID': str})
required = {'Image Index', 'Finding Labels', 'Patient ID'}
missing_columns = required - set(metadata.columns)
if missing_columns:
    raise ValueError(f'Metadata is missing required columns: {sorted(missing_columns)}')
if metadata['Image Index'].duplicated().any():
    raise ValueError('Image Index contains duplicate rows')

unknown_labels = set()
for value in metadata['Finding Labels'].dropna().astype(str):
    unknown_labels.update(set(value.split('|')) - set(LABELS) - {'No Finding'})
if unknown_labels:
    raise ValueError(f'Unexpected labels: {sorted(unknown_labels)}')
if metadata[['Image Index', 'Finding Labels', 'Patient ID']].isna().any().any():
    raise ValueError('Required metadata fields contain missing values')

for label in LABELS:
    metadata[label] = metadata['Finding Labels'].map(
        lambda value: int(label in str(value).split('|'))
    ).astype('uint8')
metadata['Patient ID'] = metadata['Patient ID'].astype(str).str.strip()
if (metadata['Patient ID'] == '').any():
    raise ValueError('Patient ID contains an empty value')

# Search the NIH image folders beside the metadata instead of recursively indexing
# every PNG under DATA_ROOT (which may be the entire /kaggle/input mount).
expected_images = set(metadata['Image Index'])
dataset_dir = metadata_path.parent
image_dirs = set()
for candidate in dataset_dir.glob('images*'):
    if candidate.is_dir():
        nested_images = candidate / 'images'
        image_dirs.add(nested_images if nested_images.is_dir() else candidate)
for candidate in dataset_dir.glob('images'):
    if candidate.is_dir():
        image_dirs.add(candidate)

# Some mirrors add one wrapper folder around the standard images_###/images layout.
if not image_dirs:
    image_dirs = {candidate for candidate in dataset_dir.rglob('images') if candidate.is_dir()}
if not image_dirs:
    raise FileNotFoundError(
        f'No images* folders found beside the metadata file at {dataset_dir}. '
        'Check that all NIH image archives were extracted.'
    )

image_paths = {}
for image_dir in sorted(image_dirs):
    print(f'Indexing {image_dir} ...')
    with os.scandir(image_dir) as entries:
        for entry in tqdm(entries, desc=image_dir.name, unit='files', leave=False):
            if entry.is_file() and entry.name in expected_images:
                if entry.name in image_paths:
                    raise ValueError(f'Duplicate image basename found: {entry.name}')
                image_paths[entry.name] = str(Path(entry.path).resolve())
metadata['path'] = metadata['Image Index'].map(image_paths)
missing_images = int(metadata['path'].isna().sum())
print(f'Metadata file: {metadata_path}')
print(f'Images in metadata: {len(metadata):,}')
print(f'Unique patients: {metadata["Patient ID"].nunique():,}')
print(f'Extracted PNG files found: {len(image_paths):,}')
print(f'Metadata rows with missing image files: {missing_images:,}')
metadata_sha256 = hashlib.sha256(metadata_path.read_bytes()).hexdigest()
dataset_info = {'metadata_file': str(metadata_path), 'metadata_sha256': metadata_sha256,
                'images': len(metadata), 'patients': int(metadata['Patient ID'].nunique()),
                'dataset_source': DATA_SOURCE, 'dataset_handle': KAGGLE_HANDLE}
dataset_info_path = OUTPUT_DIR / 'dataset_info.json'
if dataset_info_path.exists():
    previous_info = json.loads(dataset_info_path.read_text(encoding='utf-8'))
    if RESUME_TRAINING and previous_info.get('metadata_sha256') != metadata_sha256:
        raise ValueError('Metadata changed since the previous run; use a new RUN_NAME.')
dataset_info_path.write_text(json.dumps(dataset_info, indent=2), encoding='utf-8')
if missing_images:
    raise FileNotFoundError('Extract all image archives before running this full-dataset notebook.')


## 5. Use official patient-level lists and make validation patients

The test images come directly from `test_list.txt`; they stay out of training and checkpoint selection. Validation patients are sampled only from `train_val_list.txt`, stratifying patients into all-zero versus at-least-one-finding groups. This preserves patient separation, although it cannot perfectly stratify all 14 labels at once.

In [ ]:
def find_one(filename):
    matches = list(DATA_ROOT.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {filename} under {DATA_ROOT}; found {len(matches)}')
    return matches[0]

def read_image_list(filename):
    values = [line.strip() for line in find_one(filename).read_text().splitlines() if line.strip()]
    if len(values) != len(set(values)):
        raise ValueError(f'{filename} contains duplicate image names')
    return set(values)

trainval_ids = read_image_list('train_val_list.txt')
test_ids = read_image_list('test_list.txt')
if trainval_ids & test_ids:
    raise ValueError('Official train/validation and test image lists overlap')
metadata_ids = set(metadata['Image Index'])
if metadata_ids != trainval_ids | test_ids:
    raise ValueError(
        f'Official lists do not exactly cover metadata: {len(metadata_ids - (trainval_ids | test_ids))} '
        f'metadata-only, {len((trainval_ids | test_ids) - metadata_ids)} list-only image IDs.'
    )

trainval_df = metadata[metadata['Image Index'].isin(trainval_ids)].copy()
test_df = metadata[metadata['Image Index'].isin(test_ids)].copy()
patient_table = trainval_df.groupby('Patient ID')[LABELS].max()
patient_any_finding = (patient_table.sum(axis=1) > 0).astype(int)
train_patients, val_patients = train_test_split(
    patient_table.index.to_numpy(),
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=patient_any_finding.to_numpy()
)
train_df = trainval_df[trainval_df['Patient ID'].isin(train_patients)].copy()
val_df = trainval_df[trainval_df['Patient ID'].isin(val_patients)].copy()

for left_name, left, right_name, right in [
    ('train', train_df, 'validation', val_df),
    ('train', train_df, 'test', test_df),
    ('validation', val_df, 'test', test_df)
]:
    overlap = set(left['Patient ID']) & set(right['Patient ID'])
    if overlap:
        raise ValueError(f'Patient leakage between {left_name} and {right_name}: {len(overlap)} patients')

split_summary = pd.DataFrame([
    {'split': name, 'images': len(frame), 'patients': frame['Patient ID'].nunique(),
     'images_with_any_positive_label': int(frame[LABELS].any(axis=1).sum())}
    for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]
]).set_index('split')
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / 'split_summary.csv')


In [ ]:
label_audit = pd.DataFrame({
    'positive_images': train_df[LABELS].sum().astype(int),
    'train_prevalence': train_df[LABELS].mean(),
    'validation_positive_images': val_df[LABELS].sum().astype(int),
    'test_positive_images': test_df[LABELS].sum().astype(int),
}).sort_values('positive_images')
display(label_audit)
label_audit.to_csv(OUTPUT_DIR / 'label_support.csv')
label_audit['positive_images'].plot(kind='barh', figsize=(8, 6), title='Positive training images by finding')
plt.xlabel('Number of images with the label')
plt.tight_layout()
plt.show()


## 6. Preprocess images and create data loaders

The model uses ImageNet normalization to match its pretrained backbone. Images are resized without cropping, with padding if needed. Training uses small rotations and translations. Horizontal flipping is left out so that laterality is preserved.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def fit_to_square(image, size):
    image = ImageOps.contain(image, (size, size))
    canvas = Image.new('RGB', (size, size), color=(0, 0, 0))
    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    return canvas

class ChestXray14Dataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.training = training
        ops = [T.Lambda(lambda image: fit_to_square(image, IMAGE_SIZE))]
        if training:
            ops.append(T.RandomAffine(degrees=5, translate=(0.02, 0.02), fill=(0, 0, 0)))
        ops.extend([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        self.transform = T.Compose(ops)
        self.targets = self.frame[LABELS].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        path = self.frame.iloc[index]['path']
        with Image.open(path) as image:
            image = image.convert('RGB')
            x = self.transform(image)
        y = torch.from_numpy(self.targets[index])
        return x, y

loader_options = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': torch.cuda.is_available(),
}
train_loader = DataLoader(ChestXray14Dataset(train_df, training=True), shuffle=True, **loader_options)
val_loader = DataLoader(ChestXray14Dataset(val_df, training=False), shuffle=False, **loader_options)
test_loader = DataLoader(ChestXray14Dataset(test_df, training=False), shuffle=False, **loader_options)

example_x, example_y = next(iter(train_loader))
print('Image batch:', tuple(example_x.shape), '| label batch:', tuple(example_y.shape))
print('Number of positive labels in first batch:', int(example_y.sum().item()))


## 6a. Preview images before model input

Each row shows one training image: **original file**, **resized and padded evaluation input**, and **one augmented training input**. The last two columns come from the actual dataset class. Normalization is reversed **only for display**; the model still receives normalized three-channel tensors. The augmented view is one reproducible example, while training creates fresh augmentations. Labels below the row are dataset annotations, not model predictions.

The figure is saved as `figures/sample_preprocessing.png` and `.pdf`. Change `PREVIEW_COUNT` to show more images. This cell only displays images; it does not run the model.


In [ ]:
import textwrap

PREVIEW_COUNT = 3
preview_frame = train_df.sample(n=min(PREVIEW_COUNT, len(train_df)), random_state=SEED).reset_index(drop=True)
if preview_frame.empty:
    raise ValueError('No training images are available to preview.')
preview_eval = ChestXray14Dataset(preview_frame, training=False)
preview_train = ChestXray14Dataset(preview_frame, training=True)
preview_mean = np.asarray(IMAGENET_MEAN, dtype=np.float32)[None, None, :]
preview_std = np.asarray(IMAGENET_STD, dtype=np.float32)[None, None, :]

def tensor_for_display(tensor):
    pixels = tensor.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(pixels * preview_std + preview_mean, 0, 1)

fig, axes = plt.subplots(len(preview_frame), 3, figsize=(12, 4.3 * len(preview_frame)), squeeze=False)
# RandomAffine draws on the CPU generator; restore it so previewing does not alter training randomness.
with torch.random.fork_rng(devices=[]):
    torch.random.default_generator.manual_seed(SEED)
    for row_index, row in preview_frame.iterrows():
        with Image.open(row['path']) as original:
            original_mode, original_size = original.mode, original.size
            original_rgb = np.array(original.convert('RGB'))
        eval_tensor, _ = preview_eval[row_index]
        train_tensor, target = preview_train[row_index]
        views = [original_rgb, tensor_for_display(eval_tensor), tensor_for_display(train_tensor)]
        titles = [
            f'Original | {original_size[0]} × {original_size[1]} | {original_mode}',
            f'Resized input | {IMAGE_SIZE} × {IMAGE_SIZE} × 3',
            'Augmented training input',
        ]
        names = [name.replace('_', ' ') for name, present in zip(LABELS, target.tolist()) if present]
        annotation = ', '.join(names) if names else 'No Finding (all 14 targets are zero)'
        for axis, pixels, title in zip(axes[row_index], views, titles):
            axis.imshow(pixels)
            axis.set_title(title, fontsize=11)
            axis.set_xticks([])
            axis.set_yticks([])
        axes[row_index, 0].set_xlabel(row['Image Index'], fontsize=9)
        axes[row_index, 1].set_xlabel(textwrap.fill('Dataset labels: ' + annotation, width=46), fontsize=9)
        axes[row_index, 2].set_xlabel('Normalization reversed for display only', fontsize=9)
fig.suptitle('ChestX-ray14 — sample preprocessing', fontsize=15)
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(FIGURES_DIR / 'sample_preprocessing.png', dpi=300, bbox_inches='tight')
fig.savefig(FIGURES_DIR / 'sample_preprocessing.pdf', bbox_inches='tight')
plt.show()
plt.close(fig)
print('Preview saved to:', FIGURES_DIR / 'sample_preprocessing.png')


## 7. Build DenseNet-121 with channel and category-specific spatial attention

CheXNet supplies the ImageNet-pretrained DenseNet-121 backbone. The compact head borrows the attention idea from CRAL and A³Net:

- a squeeze-and-excitation style channel gate reweights feature channels;
- a separate spatial attention map is learned for each of the 14 findings;
- each map pools the feature grid into a class-specific representation before its logit is computed.

This is intentionally a small, understandable adaptation. It is not the exact CRAL residual attention block or A³Net's three-scale attention and multi-scale ensemble.

In [ ]:
class ChannelGate(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 32)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return features * self.mlp(self.pool(features))

class CategoryAttentionHead(nn.Module):
    def __init__(self, channels, n_labels, dropout=0.2):
        super().__init__()
        self.channel_gate = ChannelGate(channels)
        self.attention = nn.Conv2d(channels, n_labels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
        self.class_weight = nn.Parameter(torch.empty(n_labels, channels))
        self.class_bias = nn.Parameter(torch.zeros(n_labels))
        nn.init.xavier_uniform_(self.class_weight)

    def forward(self, features):
        features = self.channel_gate(features)
        batch, channels, height, width = features.shape
        attention_logits = self.attention(features).reshape(batch, len(LABELS), height * width)
        attention = torch.softmax(attention_logits, dim=-1)
        flat_features = features.flatten(2).transpose(1, 2)  # [batch, locations, channels]
        pooled = torch.bmm(attention, flat_features)          # [batch, labels, channels]
        pooled = self.dropout(pooled)
        logits = (pooled * self.class_weight.unsqueeze(0)).sum(dim=-1) + self.class_bias
        return logits, attention.reshape(batch, len(LABELS), height, width)

class DenseNet121CategoryAttention(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = DenseNet121_Weights.DEFAULT if pretrained else None
        backbone = densenet121(weights=weights)
        self.features = backbone.features
        self.head = CategoryAttentionHead(channels=1024, n_labels=len(LABELS))

    def forward(self, x):
        features = self.features(x)
        features = F.relu(features, inplace=True)
        return self.head(features)

model = DenseNet121CategoryAttention(pretrained=True).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


## 7a. Audit source channels and DenseNet inputs

NIH chest radiographs contain grayscale image information. The loader uses `image.convert('RGB')` to supply the three channels required by the ImageNet-pretrained DenseNet. For a grayscale source, R, G and B have equal values before normalization. ImageNet normalization uses a different mean and standard deviation per channel, so equality is not expected after normalization.

This check reads up to eight validation images using the actual dataset loader. It reports original file modes, shapes and value ranges; checks the input channel count against the model; and reverses normalization to check RGB ordering and intensity scaling. It is a sample audit, not a scan of all images. No training or model forward pass is needed. Run it after the data-loader and model-definition cells.

DenseNet's documented normalization: [Torchvision DenseNet-121](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.densenet121.html).


In [ ]:
# Small, deterministic audit of actual files; works with NUM_WORKERS = 0.
channel_audit_frame = val_df.sample(n=min(8, len(val_df)), random_state=SEED).reset_index(drop=True)
if channel_audit_frame.empty:
    raise ValueError('No validation images are available for the channel audit.')
channel_audit_dataset = ChestXray14Dataset(channel_audit_frame, training=False)
channel_audit_rows = []
channel_audit_mean = np.asarray(IMAGENET_MEAN, dtype=np.float32)[:, None, None]
channel_audit_std = np.asarray(IMAGENET_STD, dtype=np.float32)[:, None, None]
expected_input_channels = model.features.conv0.in_channels

for audit_index, audit_row in channel_audit_frame.iterrows():
    with Image.open(audit_row['path']) as source_image:
        source_mode = source_image.mode
        source_array = np.asarray(source_image)
        source_shape = tuple(source_array.shape)
        source_dtype = str(source_array.dtype)
        source_min, source_max = float(source_array.min()), float(source_array.max())
        if source_array.dtype != np.uint8:
            raise ValueError(
                f"{audit_row['Image Index']}: source mode={source_mode}, dtype={source_dtype}. "
                'This pipeline expects 8-bit PNGs. Inspect intensity scaling before converting to RGB.'
            )
        resized_rgb = np.asarray(
            fit_to_square(source_image.convert('RGB'), IMAGE_SIZE), dtype=np.float32
        ) / 255.0

    # Uses the same loading and normalization path as validation/test inference.
    actual_tensor, _ = channel_audit_dataset[audit_index]
    actual_array = actual_tensor.detach().cpu().numpy()
    assert actual_array.shape == (expected_input_channels, IMAGE_SIZE, IMAGE_SIZE), (
        f'Input shape {actual_array.shape} does not match the DenseNet input.'
    )
    assert np.isfinite(actual_array).all(), 'Model input contains nonfinite values.'
    restored_rgb = (actual_array * channel_audit_std + channel_audit_mean).transpose(1, 2, 0)
    np.testing.assert_allclose(restored_rgb, resized_rgb, rtol=0, atol=1e-6)
    channels_equal = bool(
        np.array_equal(resized_rgb[..., 0], resized_rgb[..., 1])
        and np.array_equal(resized_rgb[..., 1], resized_rgb[..., 2])
    )
    if source_mode == 'L':
        assert channels_equal, 'Grayscale pixels were not duplicated equally into RGB.'
    channel_audit_rows.append({
        'image': audit_row['Image Index'], 'source_mode': source_mode,
        'source_shape': str(source_shape), 'source_dtype': source_dtype,
        'source_min': source_min, 'source_max': source_max,
        'RGB_channels_equal_before_normalization': channels_equal,
        'model_input_shape': str(actual_array.shape),
        'model_input_dtype': str(actual_tensor.dtype),
        'normalization_roundtrip_max_error': float(np.abs(restored_rgb - resized_rgb).max()),
    })

display(pd.DataFrame(channel_audit_rows))
print(f'PASS: {len(channel_audit_rows)} sampled images match DenseNet input channels ({expected_input_channels}).')
print('RGB order and [0, 1] scaling verified by reversing ImageNet normalization.')
print('Normalized channels can differ even when the original image is grayscale.')


## 7b. Quick check before full training

Run the notebook **only through this section**, then stop before section 8 to check the pipeline without starting full training. Section 7a checks source channels; this section runs **two training batches and one validation batch** on a temporary model copy. It checks input/label shapes, finite predictions and loss, attention normalization, gradients, optimizer updates, and validation inference. It uses the configured batch size, AMP, loss weighting and learning rate, with `num_workers=0` for Colab compatibility.

The check leaves the main model and saved checkpoints unchanged, and restores random-number states. It still requires the earlier dataset setup, image index, split and model cells. Timing includes startup and is not an epoch-speed benchmark. CUDA peak memory includes the main model plus the temporary copy. A pass demonstrates that these sampled batches work, not that the model is accurate or every image is readable. F1/AUROC on these tiny samples would not be meaningful.

Leave `RUN_QUICK_CHECK=True` to run it, or set it to `False` to skip on subsequent runs. Running **all** cells still proceeds to full training in section 8.


In [ ]:
RUN_QUICK_CHECK = True
QUICK_TRAIN_BATCHES = 2
QUICK_BATCH_SIZE = BATCH_SIZE  # same batch size as full training; lower only for debugging

def run_quick_pipeline_check():
    import copy
    import gc
    import time

    if QUICK_TRAIN_BATCHES < 1 or QUICK_BATCH_SIZE < 1:
        raise ValueError('Quick-check batch count and batch size must be positive.')
    if len(train_df) < QUICK_TRAIN_BATCHES * QUICK_BATCH_SIZE or len(val_df) < QUICK_BATCH_SIZE:
        raise ValueError('Not enough images for the requested quick-check batches.')
    py_rng = random.getstate()
    np_rng = np.random.get_state()
    cuda_devices = [DEVICE.index if DEVICE.index is not None else torch.cuda.current_device()] if DEVICE.type == 'cuda' else []
    temporary_model = temporary_optimizer = temporary_scaler = None
    images = targets = logits = attention = loss = None
    started = time.perf_counter()
    try:
        with torch.random.fork_rng(devices=cuda_devices):
            random.seed(SEED)
            np.random.seed(SEED)
            torch.manual_seed(SEED)
            quick_train_frame = train_df.sample(
                n=QUICK_TRAIN_BATCHES * QUICK_BATCH_SIZE, random_state=SEED
            )
            quick_val_frame = val_df.sample(n=QUICK_BATCH_SIZE, random_state=SEED)
            quick_train_loader = DataLoader(
                ChestXray14Dataset(quick_train_frame, training=True),
                batch_size=QUICK_BATCH_SIZE, shuffle=False, num_workers=0,
                pin_memory=(DEVICE.type == 'cuda'),
            )
            quick_val_loader = DataLoader(
                ChestXray14Dataset(quick_val_frame, training=False),
                batch_size=QUICK_BATCH_SIZE, shuffle=False, num_workers=0,
                pin_memory=(DEVICE.type == 'cuda'),
            )
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize(DEVICE)
                torch.cuda.reset_peak_memory_stats(DEVICE)
            temporary_model = copy.deepcopy(model).to(DEVICE)
            temporary_optimizer = torch.optim.AdamW(
                (p for p in temporary_model.parameters() if p.requires_grad),
                lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
            )
            temporary_scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))
            quick_pos_weight = None
            if USE_POS_WEIGHT:
                positives = train_df[LABELS].sum().to_numpy(dtype=np.float32)
                weights = np.clip((len(train_df) - positives) / np.maximum(positives, 1), 1, MAX_POS_WEIGHT)
                quick_pos_weight = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
            quick_criterion = nn.BCEWithLogitsLoss(pos_weight=quick_pos_weight)
            initial_head = temporary_model.head.class_weight.detach().clone()

            def check_batch(batch):
                x, y = batch
                assert tuple(x.shape[1:]) == (temporary_model.features.conv0.in_channels, IMAGE_SIZE, IMAGE_SIZE)
                assert tuple(y.shape) == (len(x), len(LABELS))
                assert x.dtype == torch.float32 and y.dtype == torch.float32
                assert torch.isfinite(x).all().item(), 'Nonfinite image input.'
                assert ((y == 0) | (y == 1)).all().item(), 'Targets must be binary.'
                return x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)

            def check_predictions(output, maps, batch_count):
                assert tuple(output.shape) == (batch_count, len(LABELS))
                assert tuple(maps.shape[:2]) == (batch_count, len(LABELS))
                assert torch.isfinite(output).all().item(), 'Nonfinite logits.'
                assert torch.isfinite(maps).all().item(), 'Nonfinite attention.'
                assert (maps >= 0).all().item(), 'Negative attention weights.'
                torch.testing.assert_close(
                    maps.float().flatten(2).sum(-1),
                    torch.ones(batch_count, len(LABELS), device=DEVICE),
                    rtol=2e-3, atol=2e-3,
                )

            temporary_model.train()
            for batch_index, batch in enumerate(quick_train_loader, start=1):
                images, targets = check_batch(batch)
                temporary_optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                    logits, attention = temporary_model(images)
                    loss = quick_criterion(logits, targets)
                check_predictions(logits, attention, len(images))
                assert torch.isfinite(loss).item(), 'Nonfinite training loss.'
                temporary_scaler.scale(loss).backward()
                temporary_scaler.unscale_(temporary_optimizer)
                gradients = [p.grad for p in temporary_model.parameters() if p.requires_grad and p.grad is not None]
                assert gradients, 'No gradients were computed.'
                assert all(torch.isfinite(g).all().item() for g in gradients), (
                    'Nonfinite gradients. AMP can overflow at its initial scale; inspect the scaler and loss.'
                )
                assert any(torch.count_nonzero(g).item() > 0 for g in gradients), 'All gradients are zero.'
                temporary_scaler.step(temporary_optimizer)
                temporary_scaler.update()
                print(f'Quick train batch {batch_index}/{QUICK_TRAIN_BATCHES}: loss={loss.item():.4f}')
            assert not torch.equal(initial_head, temporary_model.head.class_weight.detach()), 'Head weights did not update.'

            temporary_model.eval()
            with torch.no_grad():
                images, targets = check_batch(next(iter(quick_val_loader)))
                # Match the current full-pipeline validation precision.
                logits, attention = temporary_model(images)
                loss = quick_criterion(logits, targets)
                check_predictions(logits, attention, len(images))
                assert torch.isfinite(loss).item(), 'Nonfinite validation loss.'
                probabilities = torch.sigmoid(logits)
                assert ((probabilities >= 0) & (probabilities <= 1)).all().item()
                print(f'Quick validation batch: loss={loss.item():.4f}')
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize(DEVICE)
                print('GPU:', torch.cuda.get_device_name(DEVICE))
                print(f'Peak CUDA allocated memory (including both model copies): {torch.cuda.max_memory_allocated(DEVICE) / 1024**3:.2f} GiB')
            print(f'PASS: {QUICK_TRAIN_BATCHES} training batches + 1 validation batch in {time.perf_counter() - started:.1f}s.')
            print('Main model and checkpoint files are unchanged. Continue to section 8 when ready.')
    finally:
        random.setstate(py_rng)
        np.random.set_state(np_rng)
        # Function-local temporary tensors are released on return, including on failure.
        temporary_optimizer = temporary_scaler = temporary_model = None
        gc.collect()

if RUN_QUICK_CHECK:
    run_quick_pipeline_check()
else:
    print('Quick check skipped.')


## 8. Train and choose the checkpoint on validation data

The output is 14 logits; `BCEWithLogitsLoss` applies a separate binary loss to each disease and allows co-occurrence. The training loop tracks per-label AUROC and average precision. The best checkpoint is chosen by validation macro AUROC. The test set is not used here.

At final evaluation, the notebook reports ranking metrics (AUROC and average precision) plus threshold-based metrics (precision, recall, F1, specificity, and accuracy). It reports a fixed 0.5 threshold and a per-label threshold selected to maximize F1 on validation data; the test set is never used to choose thresholds.

Training artifacts are written into the selected run folder. `last_checkpoint.pt` is updated after every completed epoch for resuming. `best_densenet121_category_attention.pt` stores the validation-selected model. The notebook also updates `epoch_history.csv` after every epoch.

### 8a. Prediction and metric functions

This cell only defines reusable functions. It can be rerun without starting training. Precision and F1 use zero when their denominator is zero. Recall and specificity are undefined when their corresponding class is absent. Macro summaries omit undefined values; sample F1 assigns zero to an empty target and empty prediction.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

@torch.no_grad()
def collect_predictions(loader):
    model.eval()
    targets_all, probabilities_all = [], []
    total_loss, total_examples = 0.0, 0
    for images, targets in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        logits, _ = model(images)
        loss = criterion(logits, targets)
        probabilities = torch.sigmoid(logits)
        total_loss += loss.item() * len(images)
        total_examples += len(images)
        targets_all.append(targets.cpu().numpy())
        probabilities_all.append(probabilities.cpu().numpy())
    y_true = np.concatenate(targets_all, axis=0)
    y_score = np.concatenate(probabilities_all, axis=0)
    return total_loss / max(total_examples, 1), y_true, y_score

def multilabel_metrics(y_true, y_score):
    aucs, aps = [], []
    rows = []
    for index, label in enumerate(LABELS):
        y = y_true[:, index]
        auc = roc_auc_score(y, y_score[:, index]) if np.unique(y).size == 2 else np.nan
        ap = average_precision_score(y, y_score[:, index]) if y.sum() > 0 else np.nan
        aucs.append(auc)
        aps.append(ap)
        rows.append({'label': label, 'AUROC': auc, 'Average precision': ap,
                     'positive examples': int(y.sum()), 'prevalence': float(y.mean())})
    return float(np.nanmean(aucs)), float(np.nanmean(aps)), pd.DataFrame(rows)


def select_f1_thresholds(y_true, y_score):
    """Choose one F1-maximizing probability threshold per label on validation data."""
    from sklearn.metrics import precision_recall_curve
    thresholds = np.full(len(LABELS), 0.5, dtype=np.float32)
    for index in range(len(LABELS)):
        y = y_true[:, index].astype(np.uint8)
        if np.unique(y).size < 2:
            continue  # no useful threshold can be selected when validation has one class
        precision, recall, candidates = precision_recall_curve(y, y_score[:, index])
        denominator = precision[:-1] + recall[:-1]
        f1 = np.divide(
            2 * precision[:-1] * recall[:-1], denominator,
            out=np.zeros_like(denominator), where=denominator > 0
        )
        thresholds[index] = float(candidates[int(np.argmax(f1))])
    return thresholds


def threshold_metrics(y_true, y_score, thresholds):
    """Return per-label and aggregate classification metrics for fixed thresholds."""
    y_true = np.asarray(y_true, dtype=np.uint8)
    y_pred = (np.asarray(y_score) >= np.asarray(thresholds).reshape(1, -1)).astype(np.uint8)
    tp = np.sum((y_true == 1) & (y_pred == 1), axis=0).astype(float)
    fp = np.sum((y_true == 0) & (y_pred == 1), axis=0).astype(float)
    tn = np.sum((y_true == 0) & (y_pred == 0), axis=0).astype(float)
    fn = np.sum((y_true == 1) & (y_pred == 0), axis=0).astype(float)

    def divide(numerator, denominator, zero_value=0.0):
        return np.divide(
            numerator, denominator,
            out=np.full(np.shape(numerator), zero_value, dtype=float),
            where=np.asarray(denominator) != 0
        )

    precision = divide(tp, tp + fp)
    recall = divide(tp, tp + fn, zero_value=np.nan)
    specificity = divide(tn, tn + fp, zero_value=np.nan)
    f1 = divide(2 * tp, 2 * tp + fp + fn)
    accuracy = divide(tp + tn, tp + fp + tn + fn)
    balanced_accuracy = np.where(
        np.isfinite(recall) & np.isfinite(specificity),
        (recall + specificity) / 2,
        np.nan
    )
    per_label = pd.DataFrame({
        'precision': precision, 'recall': recall, 'f1': f1,
        'specificity': specificity, 'accuracy': accuracy,
        'balanced_accuracy': balanced_accuracy,
        'true_positive': tp.astype(int), 'false_positive': fp.astype(int),
        'true_negative': tn.astype(int), 'false_negative': fn.astype(int),
    }, index=LABELS).rename_axis('label').reset_index()

    tp_all, fp_all = float(tp.sum()), float(fp.sum())
    tn_all, fn_all = float(tn.sum()), float(fn.sum())
    micro_precision = float(divide(tp_all, tp_all + fp_all))
    micro_recall = float(divide(tp_all, tp_all + fn_all, zero_value=np.nan))
    micro_specificity = float(divide(tn_all, tn_all + fp_all, zero_value=np.nan))
    micro_f1 = float(divide(2 * tp_all, 2 * tp_all + fp_all + fn_all))
    micro_accuracy = float(divide(tp_all + tn_all, tp_all + fp_all + tn_all + fn_all))
    if np.isfinite(micro_recall) and np.isfinite(micro_specificity):
        micro_balanced_accuracy = (micro_recall + micro_specificity) / 2
    else:
        micro_balanced_accuracy = np.nan

    sample_tp = np.sum((y_true == 1) & (y_pred == 1), axis=1)
    sample_fp = np.sum((y_true == 0) & (y_pred == 1), axis=1)
    sample_fn = np.sum((y_true == 1) & (y_pred == 0), axis=1)
    sample_f1 = divide(2 * sample_tp, 2 * sample_tp + sample_fp + sample_fn)
    metric_columns = ['precision', 'recall', 'f1', 'specificity', 'accuracy', 'balanced_accuracy']
    summary = {f'macro_{name}': float(np.nanmean(per_label[name])) for name in metric_columns}
    summary.update({
        'micro_precision': micro_precision,
        'micro_recall': micro_recall,
        'micro_f1': micro_f1,
        'micro_specificity': micro_specificity,
        'micro_accuracy': micro_accuracy,
        'micro_balanced_accuracy': float(micro_balanced_accuracy),
        'samples_f1': float(np.mean(sample_f1)),
        'exact_match_accuracy': float(np.mean(np.all(y_true == y_pred, axis=1))),
        'hamming_loss': float(np.mean(y_true != y_pred)),
    })
    return per_label, summary



### 8b. Full training

Run this cell after the quick check passes. Model selection uses validation macro AUROC.


In [ ]:
train_positive = train_df[LABELS].sum(axis=0).to_numpy(dtype=np.float32)
train_negative = len(train_df) - train_positive
positive_weights = np.clip(train_negative / np.maximum(train_positive, 1.0), 1.0, MAX_POS_WEIGHT)
positive_weights = torch.tensor(positive_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weights if USE_POS_WEIGHT else None)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=1
)


history = []
best_val_auc = -np.inf
start_epoch = 1
best_path = OUTPUT_DIR / 'best_densenet121_category_attention.pt'
last_path = OUTPUT_DIR / 'last_checkpoint.pt'
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

if RESUME_TRAINING:
    if not last_path.exists():
        raise FileNotFoundError(f'No epoch checkpoint to resume from: {last_path}')
    saved = torch.load(last_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(saved['model_state_dict'])
    optimizer.load_state_dict(saved['optimizer_state_dict'])
    scheduler.load_state_dict(saved['scheduler_state_dict'])
    scaler.load_state_dict(saved['scaler_state_dict'])
    history = saved['history']
    best_val_auc = saved['best_val_auroc']
    start_epoch = saved['epoch'] + 1
    print(f'Resuming after completed epoch {saved["epoch"]}; best validation AUROC={best_val_auc:.4f}')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss, seen = 0.0, 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for images, targets in progress:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            logits, _ = model(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * len(images)
        seen += len(images)
        progress.set_postfix(loss=f'{running_loss / max(seen, 1):.4f}')

    train_loss = running_loss / max(seen, 1)
    val_loss, y_val, p_val = collect_predictions(val_loader)
    val_auc, val_ap, _ = multilabel_metrics(y_val, p_val)
    scheduler.step(val_loss)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
           'val_macro_auroc': val_auc, 'val_macro_average_precision': val_ap,
           'learning_rate': optimizer.param_groups[0]['lr']}
    history.append(row)
    print(row)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'labels': LABELS,
            'image_size': IMAGE_SIZE,
            'use_pos_weight': USE_POS_WEIGHT,
            'validation_macro_auroc': val_auc,
            'seed': SEED,
        }, best_path)

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'history': history,
        'best_val_auroc': best_val_auc,
        'labels': LABELS,
        'run_config': RUN_CONFIG,
    }, last_path)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'epoch_history.csv', index=False)

history_df = pd.DataFrame(history)
display(history_df)
print('Best validation macro AUROC:', round(best_val_auc, 4))
print('Best checkpoint:', best_path)
print('Resume checkpoint:', last_path)
print('Run artifacts folder:', OUTPUT_DIR)


## 9. Evaluation report

Evaluation is organized into four parts: **9a** generates predictions with the selected checkpoint; **9b** creates compact tables; **9c** creates figures; **9d** saves the complete report. Rerun 9b–9d to change presentation without repeating inference or training, as long as the prediction variables remain in memory.

The official test patients are held out from training. The checkpoint is selected by validation AUROC; per-disease thresholds are selected by validation F1. Both the fixed **0.5** cutoff and the validation-selected cutoffs are reported. Average precision (AP) is reported as AP, not mislabeled as trapezoidal PR-AUC. Scores range from 0 to 1; Hamming loss is better when lower. These are point estimates from one run, without confidence intervals.

### 9a. Restore the selected checkpoint and generate predictions


In [ ]:
best_path = OUTPUT_DIR / 'best_densenet121_category_attention.pt'
checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
if checkpoint['labels'] != LABELS:
    raise ValueError('Checkpoint labels do not match this notebook.')
model.load_state_dict(checkpoint['model_state_dict'])

print('Selecting thresholds on validation patients...')
_, y_val_best, p_val_best = collect_predictions(val_loader)
val_thresholds = select_f1_thresholds(y_val_best, p_val_best)
val_threshold_results, _ = threshold_metrics(y_val_best, p_val_best, val_thresholds)
threshold_table = pd.DataFrame({
    'label': LABELS, 'threshold': val_thresholds,
    'validation_f1_at_threshold': val_threshold_results['f1'].to_numpy(),
    'validation_positives': y_val_best.sum(axis=0).astype(int),
    'selection': [
        'maximum validation F1' if np.unique(y_val_best[:, i]).size == 2 else 'fallback 0.5: validation has one class'
        for i in range(len(LABELS))
    ],
})
threshold_table.to_csv(OUTPUT_DIR / 'validation_f1_thresholds.csv', index=False)
np.savez_compressed(
    OUTPUT_DIR / 'validation_predictions.npz', y_true=y_val_best, y_score=p_val_best,
    labels=np.array(LABELS), image_ids=val_df['Image Index'].to_numpy(dtype=str),
    selected_epoch=np.array(checkpoint['epoch']),
)

print('Evaluating official test patients...')
test_loss, y_test, p_test = collect_predictions(test_loader)
np.savez_compressed(
    OUTPUT_DIR / 'test_predictions.npz', y_true=y_test, y_score=p_test,
    labels=np.array(LABELS), validation_f1_thresholds=val_thresholds,
    image_ids=test_df['Image Index'].to_numpy(dtype=str), selected_epoch=np.array(checkpoint['epoch']),
)
print(f"Predictions saved: epoch {checkpoint['epoch']}; {len(y_test):,} test images. Continue to the report tables.")


### 9b. Report tables

The run summary records the evaluation setup. The overall table compares the two threshold methods, and two per-disease tables separate ranking metrics from classification metrics. **Macro** gives each disease equal weight; **micro** pools decisions across all images and diseases. Label accuracy can be high when negatives dominate, so read it alongside F1, recall and balanced accuracy. Exact-match accuracy requires all 14 predictions for an image to match.

The displayed classification table uses validation-selected thresholds. The fixed-0.5 per-disease table, confusion counts, and full-precision data are also saved in `report_tables/`. Four decimal places are used only for display.


In [ ]:
from IPython.display import HTML, display
import html

REPORT_TABLE_HTML = []
REPORT_FIGURES = []
REPORT_TABLE_STYLE = """
<style>
.cxr-table {border-collapse:collapse; font-family:Arial,sans-serif; font-size:13px; margin:12px 0 26px;}
.cxr-table th {background:#16324f; color:white; text-align:right; padding:9px;}
.cxr-table td {padding:8px 10px; text-align:right; border-bottom:1px solid #dce3ec;}
.cxr-table th:first-child,.cxr-table td:first-child {text-align:left;}
.cxr-table tbody tr:nth-child(even) {background:#f3f6fa;}
.cxr-caption {font:600 17px Arial,sans-serif; color:#16324f; margin-top:22px;}
.cxr-table-wrap {overflow-x:auto;}
</style>
"""

def show_report_table(frame, title, filename):
    frame.to_csv(TABLES_DIR / (filename + '.csv'), index=False)
    table_html = frame.to_html(index=False, classes='cxr-table', border=0, escape=True,
                               float_format=lambda value: f'{value:.4f}', na_rep='—')
    fragment = '<div class="cxr-caption">' + html.escape(title) + '</div><div class="cxr-table-wrap">' + table_html + '</div>'
    (TABLES_DIR / (filename + '.html')).write_text(REPORT_TABLE_STYLE + fragment, encoding='utf-8')
    REPORT_TABLE_HTML.append(fragment)
    display(HTML(REPORT_TABLE_STYLE + fragment))

# Keep raw result filenames compatible with earlier runs; display concise tables separately.
test_macro_auc, test_macro_ap, ranking_results = multilabel_metrics(y_test, p_test)
fixed_results, fixed_summary = threshold_metrics(y_test, p_test, np.full(len(LABELS), 0.5))
tuned_results, tuned_summary = threshold_metrics(y_test, p_test, val_thresholds)
test_results = ranking_results.merge(
    fixed_results.rename(columns={c: 'fixed_0_5_' + c for c in fixed_results if c != 'label'}), on='label'
).merge(
    tuned_results.rename(columns={c: 'validation_f1_threshold_' + c for c in tuned_results if c != 'label'}), on='label'
)
test_results['validation_f1_threshold'] = val_thresholds
test_results.to_csv(OUTPUT_DIR / 'per_label_test_metrics.csv', index=False)
test_summary = pd.DataFrame([
    {'threshold_method': method, 'test_loss': test_loss, 'macro_auroc': test_macro_auc,
     'macro_average_precision': test_macro_ap, **values}
    for method, values in [('fixed_0.5', fixed_summary), ('validation_max_f1_per_label', tuned_summary)]
])
test_summary.to_csv(OUTPUT_DIR / 'test_metric_summary.csv', index=False)

run_overview = pd.DataFrame([
    ('Run', RUN_NAME), ('Model', 'ImageNet DenseNet-121 + channel/category attention; full fine-tuning'),
    ('Selected epoch', str(checkpoint['epoch'])),
    ('Validation macro AUROC at selection', f"{checkpoint['validation_macro_auroc']:.4f}"),
    ('Training images / patients', f'{len(train_df):,} / {train_df["Patient ID"].nunique():,}'),
    ('Validation images / patients', f'{len(val_df):,} / {val_df["Patient ID"].nunique():,}'),
    ('Official test images / patients', f'{len(test_df):,} / {test_df["Patient ID"].nunique():,}'),
    ('Image size / channels', f'{IMAGE_SIZE} × {IMAGE_SIZE} / 3'),
    ('Seed', str(SEED)), ('Test BCE loss', f'{test_loss:.4f}'),
    ('Threshold selection', 'Per-disease maximum F1 on validation data only'),
], columns=['Item', 'Value'])
show_report_table(run_overview, 'Table 1. Run and evaluation protocol', 'run_overview')
metric_names = [
    ('macro_auroc', 'Macro AUROC'), ('macro_average_precision', 'Macro average precision (AP)'),
    ('macro_f1', 'Macro F1'), ('micro_f1', 'Micro F1'),
    ('macro_precision', 'Macro precision'), ('micro_precision', 'Micro precision'),
    ('macro_recall', 'Macro recall / sensitivity'), ('micro_recall', 'Micro recall / sensitivity'),
    ('macro_specificity', 'Macro specificity'), ('macro_balanced_accuracy', 'Macro balanced accuracy'),
    ('macro_accuracy', 'Mean per-label accuracy'), ('samples_f1', 'Sample-averaged F1'),
    ('exact_match_accuracy', 'Exact-match accuracy'), ('hamming_loss', 'Hamming loss (lower is better)'),
]
summary_table = pd.DataFrame([
    {'Metric': name, 'Fixed 0.5': test_summary.iloc[0][key],
     'Validation-selected': test_summary.iloc[1][key]}
    for key, name in metric_names
])
show_report_table(summary_table, 'Table 2. Overall test performance', 'overall_test_performance')
ranking_table = ranking_results.rename(columns={
    'label':'Finding', 'Average precision':'AP', 'positive examples':'Positive test images', 'prevalence':'Prevalence'
}).copy()
ranking_table['Finding'] = ranking_table['Finding'].str.replace('_', ' ', regex=False)
show_report_table(ranking_table, 'Table 3. Test ranking metrics by finding', 'ranking_by_finding')

def classification_report_table(rows, thresholds):
    table = rows[['label', 'precision', 'recall', 'f1', 'specificity', 'balanced_accuracy']].copy()
    table.insert(1, 'Threshold', thresholds)
    table['label'] = table['label'].str.replace('_', ' ', regex=False)
    return table.rename(columns={
        'label':'Finding', 'precision':'Precision', 'recall':'Recall', 'f1':'F1',
        'specificity':'Specificity', 'balanced_accuracy':'Balanced accuracy'
    })
classification_table = classification_report_table(tuned_results, val_thresholds)
show_report_table(classification_table, 'Table 4. Classification at validation-selected thresholds', 'classification_validation_thresholds')
fixed_table = classification_report_table(fixed_results, np.full(len(LABELS), 0.5))
fixed_table.to_csv(TABLES_DIR / 'classification_fixed_0_5.csv', index=False)
fixed_results.to_csv(TABLES_DIR / 'confusion_counts_fixed_0_5.csv', index=False)
tuned_results.to_csv(TABLES_DIR / 'confusion_counts_validation_thresholds.csv', index=False)
threshold_table.to_csv(TABLES_DIR / 'threshold_selection.csv', index=False)


### 9c. Report figures

Figures show training history, per-disease performance, ROC curves, precision–recall curves, and confusion matrices. Each disease has its own curve panel to keep the labels readable. The PR baseline is the disease prevalence. Confusion matrices use validation-selected thresholds and are normalized within each true class; their annotations also show raw image counts. All figures are saved as 300-dpi PNG and PDF.


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

REPORT_FIGURES = []
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'figure.facecolor': 'white'})

def save_report_figure(fig, filename, caption):
    fig.savefig(FIGURES_DIR / (filename + '.png'), dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(FIGURES_DIR / (filename + '.pdf'), bbox_inches='tight', facecolor='white')
    REPORT_FIGURES.append((filename + '.png', caption))
    plt.show()
    plt.close(fig)

history_plot = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
axes[0].plot(history_plot['epoch'], history_plot['train_loss'], 'o-', label='Training')
axes[0].plot(history_plot['epoch'], history_plot['val_loss'], 'o-', label='Validation')
axes[0].set(title='Binary cross-entropy', ylabel='Loss')
axes[0].legend(frameon=False)
for axis, column, title in zip(axes[1:], ['val_macro_auroc', 'val_macro_average_precision'], ['Validation macro AUROC', 'Validation macro AP']):
    axis.plot(history_plot['epoch'], history_plot[column], 'o-', color='#176b87')
    axis.set(title=title, ylabel='Score', ylim=(0, 1))
for axis in axes:
    axis.axvline(checkpoint['epoch'], color='#bd592e', linestyle='--', linewidth=1)
    axis.set_xlabel('Epoch')
    axis.set_xticks(history_plot['epoch'])
    axis.grid(alpha=0.2)
fig.suptitle('Training history | dashed line: selected checkpoint', fontsize=14)
save_report_figure(fig, 'training_history', 'Training and validation history. The dashed line marks the checkpoint selected by validation macro AUROC.')

fig, axes = plt.subplots(1, 3, figsize=(14, 7), sharey=True, constrained_layout=True)
y_positions = np.arange(len(LABELS))
for axis, values, title in zip(axes, [ranking_results['AUROC'], ranking_results['Average precision'], tuned_results['f1']], ['AUROC', 'Average precision', 'F1 | validation-selected thresholds']):
    axis.barh(y_positions, values, color='#176b87')
    axis.set(xlim=(0, 1.12), title=title, xlabel='Score')
    axis.set_xticks([0, .25, .5, .75, 1])
    axis.grid(axis='x', alpha=.2)
    for y, value in zip(y_positions, values):
        axis.text((value if np.isfinite(value) else 0) + .02, y, f'{value:.3f}' if np.isfinite(value) else 'N/A', va='center', fontsize=8)
axes[0].set_yticks(y_positions, [label.replace('_', ' ') for label in LABELS])
axes[0].invert_yaxis()
fig.suptitle('Official test set | performance by finding', fontsize=14)
save_report_figure(fig, 'performance_by_finding', 'Per-finding test AUROC, average precision and F1. F1 uses thresholds chosen on validation data.')

for kind in ['roc', 'pr']:
    fig, axes = plt.subplots(4, 4, figsize=(13, 12), constrained_layout=True)
    for i, (axis, label) in enumerate(zip(axes.flat, LABELS)):
        truth, scores = y_test[:, i], p_test[:, i]
        if kind == 'roc' and np.unique(truth).size == 2:
            x, y, _ = roc_curve(truth, scores)
            axis.plot(x, y, color='#176b87', label=f"AUROC {ranking_results.iloc[i]['AUROC']:.3f}")
            axis.plot([0, 1], [0, 1], '--', color='#999999', linewidth=.8)
        elif kind == 'pr' and truth.sum() > 0:
            precision, recall, _ = precision_recall_curve(truth, scores)
            axis.step(recall, precision, where='post', color='#176b87', label=f"AP {ranking_results.iloc[i]['Average precision']:.3f}")
            axis.axhline(truth.mean(), linestyle='--', color='#999999', linewidth=.8)
        else:
            axis.text(.5, .5, 'Undefined for this class', ha='center', va='center')
        axis.set(title=label.replace('_', ' '), xlim=(0, 1), ylim=(0, 1.02),
                 xlabel='False positive rate' if kind == 'roc' else 'Recall',
                 ylabel='True positive rate' if kind == 'roc' else 'Precision')
        axis.grid(alpha=.15)
        if axis.get_legend_handles_labels()[0]: axis.legend(loc='lower right' if kind == 'roc' else 'upper right', fontsize=8, frameon=False)
    for axis in axes.flat[len(LABELS):]: axis.set_visible(False)
    title = 'ROC curves' if kind == 'roc' else 'Precision–recall curves'
    fig.suptitle('Official test set | ' + title, fontsize=15)
    caption = title + ('; dashed line is the chance diagonal.' if kind == 'roc' else '; dashed lines show per-disease prevalence. AP is average precision.')
    save_report_figure(fig, 'test_' + kind + '_curves', caption)

fig, axes = plt.subplots(4, 4, figsize=(12, 12), constrained_layout=True)
for i, (axis, label) in enumerate(zip(axes.flat, LABELS)):
    row = tuned_results.iloc[i]
    counts = np.array([[row['true_negative'], row['false_positive']], [row['false_negative'], row['true_positive']]], dtype=int)
    denominators = counts.sum(axis=1, keepdims=True)
    normalized = np.divide(counts, denominators, out=np.zeros_like(counts, dtype=float), where=denominators != 0)
    axis.imshow(normalized, cmap='Blues', vmin=0, vmax=1)
    for r in range(2):
        for c in range(2):
            pct = f'{normalized[r,c]:.1%}' if denominators[r,0] else 'N/A'
            axis.text(c, r, f'{counts[r,c]:,}\n{pct}', ha='center', va='center', fontsize=9,
                      color='white' if normalized[r,c] > .55 else '#16324f')
    axis.set(title=label.replace('_', ' '), xlabel='Predicted', ylabel='True')
    axis.set_xticks([0, 1], ['Negative', 'Positive'])
    axis.set_yticks([0, 1], ['Negative', 'Positive'])
for axis in axes.flat[len(LABELS):]: axis.set_visible(False)
fig.suptitle('Test confusion matrices | validation-selected thresholds', fontsize=15)
save_report_figure(fig, 'test_confusion_matrices', 'Per-disease test confusion matrices: counts and percentages within each true class. Darker blue indicates a larger row proportion; thresholds come from validation data.')


### 9d. Save the complete report

`evaluation_report.html` contains the formatted tables and embedded figures, so it can be opened independently in a browser or printed. Exact values remain available in the CSV files. The fixed-0.5 per-disease table is included as an appendix. Rerun 9b and 9c before exporting again after changing presentation.


In [ ]:
import base64
from pathlib import Path

report_images = []
preview_path = FIGURES_DIR / 'sample_preprocessing.png'
image_entries = ([('sample_preprocessing.png', 'Sample preprocessing: original, resized input, and augmented training input. Normalization is reversed only for display.')] if preview_path.exists() else []) + REPORT_FIGURES
for filename, caption in image_entries:
    encoded = base64.b64encode((FIGURES_DIR / filename).read_bytes()).decode('ascii')
    report_images.append('<figure><img src="data:image/png;base64,' + encoded + '" alt="' + html.escape(caption, quote=True) + '"><figcaption>' + html.escape(caption) + '</figcaption></figure>')
appendix = fixed_table.to_html(index=False, classes='cxr-table', border=0, escape=True,
                              float_format=lambda value: f'{value:.4f}', na_rep='—')
notes = (
    'Patient-disjoint validation and the official held-out test list are used. The checkpoint is selected by validation macro AUROC. '
    'Per-disease thresholds maximize validation F1; no threshold is selected on the test set. '
    'AUROC and AP do not depend on the classification cutoff. AP is not trapezoidal PR-AUC. '
    'Precision and F1 are set to zero when their denominator is zero. Recall/specificity are undefined when the relevant true class is absent; '
    'macro summaries omit undefined values. Sample F1 is zero for an empty target and empty prediction. '
    'All metrics are point estimates from a single seeded run; no confidence intervals were computed. '
    'Labels are report-mined NIH annotations. This simplified attention model is not an exact reproduction of a cited paper.'
)
report_html = '<!doctype html><html lang="en"><head><meta charset="utf-8"><title>ChestX-ray14 evaluation report</title>' + REPORT_TABLE_STYLE + """
<style>body{max-width:1180px;margin:40px auto;padding:0 28px;font-family:Arial,sans-serif;color:#233142;line-height:1.55}
h1,h2{color:#16324f}figure{margin:32px 0;break-inside:avoid}img{width:100%;height:auto}figcaption{font-size:13px;color:#526172}
.note{background:#f3f6fa;padding:16px;border-left:4px solid #176b87}@media print{body{margin:0;padding:8px}.cxr-table{font-size:10px}th,td{padding:5px!important}}
</style></head><body><h1>ChestX-ray14 evaluation report</h1>""" + '<p>Run: <strong>' + html.escape(RUN_NAME) + '</strong></p>'
report_html += '<p class="note">' + html.escape(notes) + '</p>'
report_html += ''.join(REPORT_TABLE_HTML) + '<h2>Figures</h2>' + ''.join(report_images)
report_html += '<h2>Appendix: classification at a fixed 0.5 threshold</h2>' + appendix + '</body></html>'
report_path = OUTPUT_DIR / 'evaluation_report.html'
report_path.write_text(report_html, encoding='utf-8')
print('Report saved:', report_path)
print('Tables:', TABLES_DIR)
print('Figures (PNG + PDF):', FIGURES_DIR)
display(HTML('<p><strong>Report ready.</strong> Open evaluation_report.html from the run folder in Google Drive. Individual tables and figures are saved alongside it.</p>'))


## 10. Visualize one class-specific attention map

This overlays the learned spatial attention weights for one finding. The attention head forces each map to sum to one, but that does **not** prove it found the disease or provide a clinically validated localization. Compare it with radiologist-drawn boxes only if you have appropriate annotations and an evaluation protocol.

In [ ]:
# Choose an image from the official test split. Change this index if desired.
DISPLAY_INDEX = 0
DISPLAY_LABEL = 'Pneumonia'
row = test_df.iloc[DISPLAY_INDEX]
dataset_item = ChestXray14Dataset(test_df.iloc[[DISPLAY_INDEX]], training=False)
image_tensor, target_tensor = dataset_item[0]
label_index = LABELS.index(DISPLAY_LABEL)

model.eval()
with torch.no_grad():
    logits, attention_maps = model(image_tensor.unsqueeze(0).to(DEVICE))
    probabilities = torch.sigmoid(logits)[0].cpu().numpy()
    attention = attention_maps[0, label_index].unsqueeze(0).unsqueeze(0)
    attention = F.interpolate(attention, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)[0, 0].cpu().numpy()

image = image_tensor.permute(1, 2, 0).cpu().numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
image = np.clip(image, 0, 1)
attention = (attention - attention.min()) / (attention.max() - attention.min() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title(f"Image | {DISPLAY_LABEL} target={int(target_tensor[label_index])}")
axes[1].imshow(image, cmap='gray')
axes[1].imshow(attention, cmap='jet', alpha=0.45)
axes[1].set_title(f"Attention | model score={probabilities[label_index]:.3f}")
for axis in axes:
    axis.axis('off')
plt.tight_layout()
plt.show()


## 11. Save and cite

The report is `evaluation_report.html`; formatted CSV/HTML tables are in `report_tables/`; 300-dpi PNG and PDF figures are in `figures/`. Source sample previews are saved before training. Section 9 also saves validation predictions, test predictions, image IDs and the selected epoch for traceability.

The final evaluation prints macro and micro classification summaries. The per-label CSV includes AUROC, average precision, precision, recall, F1, specificity, accuracy, and balanced accuracy at fixed 0.5 and validation-selected thresholds. The summary CSV also includes sample F1, exact-match accuracy, and Hamming loss. The run folder is printed near the top and contains `run_config.json`, `dataset_info.json`, split and label summaries, `last_checkpoint.pt`, `best_densenet121_category_attention.pt`, `epoch_history.csv`, `validation_f1_thresholds.csv`, `per_label_test_metrics.csv`, `test_metric_summary.csv`, and `test_predictions.npz`. These files are created as the notebook runs, so a disconnect does not erase completed epochs when the output folder is on Drive.

### Saving the notebook and continuing later

- **Google Colab:** first choose **File → Save a copy in Drive**. The notebook code and displayed cell outputs are saved in that copy; model and metrics files go to `MyDrive/ChestXrayProject/<RUN_NAME>/`. If the runtime stops, reopen the saved copy, keep the same `RUN_NAME`, set `RESUME_TRAINING=True`, and run cells from the top.
- **Kaggle Notebook:** add the dataset with **Add Input**, use `DATA_SOURCE='kaggle_input'`, then use **Save Version** after running. Files written under `/kaggle/working/<RUN_NAME>/` are notebook outputs; download them from the notebook Output panel or save a version to preserve them.
- **Local Jupyter:** save the `.ipynb` with Ctrl/Cmd+S. Results are written under `chestxray14_densenet_attention_seed42_v2_20261027/` in the working directory unless you change `RUN_NAME` or the output path.

After changing model settings, use a new `RUN_NAME` for a clean experiment. To resume, keep the same settings and name, set `RESUME_TRAINING=True`, and increase `EPOCHS` only if you want additional epochs. The notebook file and the saved model/results are separate artifacts; save both if you want the code, displayed outputs, and trained model together.

### Reading list

- Wang, X. et al. (2017). [ChestX-ray8: Hospital-scale Chest X-Ray Database and Benchmarks on Weakly-Supervised Classification and Localization of Common Thorax Diseases](https://openaccess.thecvf.com/content_cvpr_2017/html/Wang_ChestX-ray8_Hospital-Scale_Chest_CVPR_2017_paper.html). CVPR.
- Yao, L. et al. (2017). [Learning to diagnose from scratch by exploiting dependencies among labels](https://arxiv.org/abs/1710.10501).
- Rajpurkar, P. et al. (2017). [CheXNet: Radiologist-Level Pneumonia Detection on Chest X-Rays with Deep Learning](https://arxiv.org/abs/1711.05225).
- Ge, Z. et al. (2018). [Chest X-rays Classification: A Multi-Label and Fine-Grained Problem](https://arxiv.org/abs/1807.07247).
- Guan, Q. & Huang, Y. (2020). [Multi-label chest X-ray image classification via category-wise residual attention learning](https://doi.org/10.1016/j.patrec.2018.10.027). Pattern Recognition Letters.
- Wang, H. et al. (2021). [Triple attention learning for classification of 14 thoracic diseases using chest radiography](https://doi.org/10.1016/j.media.2020.101846). Medical Image Analysis.

### Interpretation limits

- This is a research-learning exercise, not a medical device or diagnostic tool.
- ChestX-ray14's report-mined labels have known noise and do not provide complete ground truth for every finding.
- AUROC values from this run are meaningful only with the exact split, preprocessing, labels, and model recorded; do not compare directly with papers unless their protocol is matched.
- Attention visualization is a model inspection aid, not evidence that a highlighted region contains pathology.
- Do not tune thresholds, settings, or epochs based on the final test set.